# Показ 3. Тематическое моделирование экзит-интервью

Запускаем модели на синтетическом датасете и смотрим, какие темы (причины увольнения) они находят.

1. **LDA** — классическая вероятностная модель, которую предлагает кейс.
2. **NMF** — разложение матрицы TF-IDF.
3. **e5-instruct + HDBSCAN** — смысловые векторы нейросети и кластеризация (подход BERTopic).

Для каждой модели считаем совпадение с эталонными причинами (ARI, F1) — датасет синтетический, поэтому правильный ответ известен.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 100)

from src.evaluation.model_comparison import load_corpus
from src.evaluation.metrics import external_metrics, majority_mapping, per_label_report
from src.models import LDATopicModel, NMFTopicModel, EmbeddingTopicModel, Encoder

texts, tokens, y = load_corpus()  # очистка, лемматизация, без пустых ответов и дублей
print(f'Ответов: {len(texts)}')
pd.DataFrame({'текст для e5': texts[:3], 'леммы для LDA/NMF': [' '.join(t) for t in tokens[:3]]})

Ответов: 459


,текст для e5,леммы для LDA/NMF
0,"Нет менторства и наставничества, новые навыки получить негде.",менторство наставничество новый навык получить негде
1,Выгорела полностью. Не могу больше.,выгореть полностью
2,"Не обеспечили нормальными условиями для удалённой работы — ни техники, ни компенсации.",обеспечить нормальный условие удаленка техника компенсация


In [2]:
def show(result, name):
    """Темы модели: размер, ключевые слова и какой эталонной причине тема соответствует."""
    m = external_metrics(y, result.labels)
    print(f"{name}: тем {result.n_topics}, ARI {m['ARI']:.2f}, macro-F1 {m['macro_F1']:.2f}")
    info = result.topic_info(8)
    info['эталон'] = info['topic'].map(majority_mapping(y, result.labels))
    return info[['topic', 'size', 'эталон', 'keywords']]

## 1. LDA

Каждый ответ — смесь тем, каждая тема — распределение слов. Обучается на частотах лемм.

In [3]:
lda = LDATopicModel(n_topics=10).fit(texts, tokens)
show(lda, 'LDA')

LDA: тем 10, ARI 0.02, macro-F1 0.17


,topic,size,эталон,keywords
0,4,65,management,"руководитель, время, задача, отдел, процесс, сделать, обещать, постоянный"
1,9,49,career,"руководитель, много, новый, команда, расти, направление, понять, задача"
2,8,48,personal,"новый, час, почти, команда, хватать, личный, жизнь, начальник"
3,6,47,conditions,"плохой, склад, коллектив, условие, команда, постоянный, время, техника"
4,1,45,schedule,"график, руководство, постоянный, год, подходить, невозможно, менять, обстоятельство"
5,5,44,schedule,"офис, сторона, удаленка, рост, режим, здоровье, город, отменить"
6,0,43,content,"коллега, система, проект, дать, план, день, ребенок, идти"
7,3,43,personal,"день, пять, согласование, задача, должность, оборудование, неделя, повышение"
8,2,42,salary,"зарплата, нормальный, смена, компенсация, найти, проблема, нельзя, год"
9,7,33,salary,"деньга, переработка, позиция, платить, совсем, ощущение, команда, рабочий"


Слова внутри тем выглядят связно, но темы почти не совпадают с настоящими причинами: в ответе в среднем 5 лемм, и LDA не хватает данных, чтобы оценить смесь тем.

## 2. NMF

In [4]:
nmf = NMFTopicModel(n_topics=10).fit(texts, tokens)
show(nmf, 'NMF')

NMF: тем 10, ARI 0.10, macro-F1 0.24


,topic,size,эталон,keywords
0,9,74,schedule,"график, здоровье, подходить, личный, склад, смена, час, постоянный"
1,6,58,team,"команда, новый, почти, общение, старый, комфортно, больший, найти"
2,4,56,salary,"зарплата, платить, устраивать, нормальный, должность, вырасти, расти, собеседование"
3,5,52,content,"задача, расти, творческий, ответственность, сделать, соответствовать, рост, ощущение"
4,1,46,schedule,"день, рабочий, неделя, однообразный, офис, пять, удаленка, коллега"
5,0,43,management,"руководитель, сотрудник, ошибка, коллектив, повышение, деньга, процесс, поощрять"
6,7,37,career,"отдел, обещать, начальник, постоянный, принимать, любой, информация, ждать"
7,8,37,management,"руководство, сторона, коллега, постоянный, тяжело, процесс, семейный, обстоятельство"
8,3,35,personal,"время, год, обещать, техника, сделать, чувствовать, поступить, забирать"
9,2,21,team,"плохой, условие, коллектив, склад, холодно, начальник, труд, сквозняк"


## 3. Эмбеддинги e5-instruct + HDBSCAN

Нейросеть `multilingual-e5-large-instruct` превращает ответ в вектор смысла. Инструкция «Identify the main reason why the employee is leaving the company» говорит модели обращать внимание на причину ухода, а не на стиль ответа. HDBSCAN сам находит число тем. Ключевые слова — c-TF-IDF по леммам.

In [5]:
e5 = EmbeddingTopicModel(encoder=Encoder('intfloat/multilingual-e5-large-instruct'),
                         clustering='hdbscan', umap_dims=5)
best = e5.fit(texts, tokens)
show(best, 'e5-instruct + HDBSCAN')

e5-instruct + HDBSCAN: тем 11, ARI 0.51, macro-F1 0.63


,topic,size,эталон,keywords
0,3,94,team,"руководитель, коллектив, коллега, команда, руководство, начальник, атмосфера, сотрудник"
1,7,58,processes,"процесс, согласование, система, меняться, задача, месяц, три, отдел"
2,6,56,content,"задача, делать, учиться, проект, рутина, сфера, интересный, попробовать"
3,4,55,salary,"зарплата, платить, деньга, дмс, бонус, оплата, оклад, получать"
4,5,51,career,"рост, повышение, обещать, развитие, год, обучение, позиция, карьерный"
5,10,51,personal,"ребенок, переезжать, город, семья, здоровье, уезжать, семейный, обстоятельство"
6,9,36,schedule,"график, смена, выходной, рабочий, норма, час, подходить, переработка"
7,2,21,conditions,"плохой, условие, болеть, тесный, холодно, сквозняк, шум, помещение"
8,0,14,schedule,"удаленка, нельзя, офис, формат, неделя, нормальный, город, день"
9,1,12,schedule,"дорога, офис, транспорт, час, добираться, промзона, почти, обед"


Типичные ответы каждой темы — то, что HR увидит в интерфейсе:

In [6]:
for t in best.topic_info()['topic']:
    print(f'Тема {t}:', ' | '.join(best.representative_docs(texts, t, 2)))

Тема 3: Руководитель натравливает сотрудников друг на друга, поощряет доносы. В коллективе из-за этого все друг другу не доверяют. | Руководитель поощряет конфликты между сотрудниками, в итоге коллектив расколот на группировки.
Тема 7: Нет единой системы постановки задач, всё в чатах, почте и устно, постоянно что-то теряется! | Хаос в процессах, никто не знает, кто за что отвечает.
Тема 6: Рутина, нет никаких интересных задач. | Интересных проектов нет, всё по шаблону.
Тема 4: С учётом инфляции я фактически получаю меньше, чем два года назад. | Платят меньше, чем обещали на собеседовании. Разница вскрылась уже после первой зарплаты.
Тема 5: Получил второе высшее, но компания никак это не учла, остался на той же позиции. | Руководитель не рассматривает меня на повышение, говорит, что я ещё не готова, но не объясняет, чего не хватает.
Тема 10: Выхожу замуж и переезжаю к мужу в Новосибирск. | Ухожу в декрет и после него планирую сидеть с ребёнком.
Тема 9: Перевели на график с ночными смен

Качество по каждой причине:

In [7]:
per_label_report(y, best.labels).round(2)

,precision,recall,f1,support
label,,,,
career,0.75,0.84,0.79,45
conditions,1.00,0.49,0.66,43
content,0.70,0.85,0.76,46
management,0.00,0.00,0.00,46
personal,0.76,0.85,0.80,46
processes,0.67,0.87,0.76,45
salary,0.80,0.88,0.84,50
schedule,0.61,0.84,0.71,45
team,0.47,0.98,0.63,45


## 4. Сравнение всех моделей

Полная таблица: `python -m src.evaluation.model_comparison` (результаты в `docs/results/`). Устойчивость — средний ARI между пятью запусками на разных 80% данных.

In [8]:
res = pd.read_csv('../docs/results/models_k10.csv')
res.sort_values('ARI', ascending=False)[['модель', 'тем', 'ARI', 'macro_F1', 'c_v', 'diversity', 'stability']].round(3)

,модель,тем,ARI,macro_F1,c_v,diversity,stability
6,multilingual-e5-large-instruct + hdbscan + UMAP5,11,0.510,0.629,0.360,0.946,0.409
4,multilingual-e5-large-instruct + kmeans,10,0.424,0.624,0.382,0.950,0.511
5,multilingual-e5-large-instruct + agglomerative,10,0.313,0.487,0.370,0.950,0.506
18,FRIDA + hdbscan + UMAP5,14,0.292,0.566,0.388,0.879,0.476
9,multilingual-e5-large + hdbscan + UMAP5,14,0.270,0.505,0.406,0.886,0.452
15,USER-bge-m3 + hdbscan + UMAP5,9,0.262,0.390,0.412,0.922,0.340
16,FRIDA + kmeans,10,0.253,0.392,0.398,0.940,0.594
17,FRIDA + agglomerative,10,0.237,0.340,0.379,0.920,0.506
14,USER-bge-m3 + agglomerative,10,0.230,0.436,0.391,0.920,0.401
13,USER-bge-m3 + kmeans,10,0.226,0.416,0.390,0.940,0.352


In [9]:
from src.evaluation.plots import plot_coherence_vs_ari
plot_coherence_vs_ari(res);

## Выводы

- Классические модели «мешка слов» на коротких ответах почти не находят реальные причины: «мало платят» и «низкий оклад» не имеют общих слов.
- Эмбеддинги решают эту проблему. Инструкция для e5 удваивает качество (ARI 0.21 → 0.42 с KMeans), а HDBSCAN доводит его до 0.51.
- Связность тем (c_v) не отражает реального качества: у LDA она самая высокая. Поэтому модель выбираем по совпадению с эталоном на синтетике, а связность используем только как дополнительный сигнал.